# 02 — Preprocessing and Feature Engineering**Step 4 of the brief.** Turn the raw CSV into three model-ready matrices — one forclassification, one for regression, one for clustering. They are **not the same matrix**,and understanding why is the point of this notebook.

In [1]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))   # [so `from src...` works from notebooks/]

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from src.config import *

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)
pd.set_option("display.max_columns", 40)

## Run the pipeline

In [2]:
from src.data_prep import build

df = build()
display(df.head())

--- CLEANING REPORT ---
  missing_values             0
  duplicates_removed         146
  zero_dimension_rows        19
  absurd_outliers_removed    9
  rows_before                53940
  rows_after                 53766
  retained                   99.68%

Saved -> /Volumes/HDD Org2/GITHUB_CLONES/capstone_diamonds_predictor_application/capstone_diamonds_predictor_application/dp/data/diamonds_clean.csv  (53766 rows x 17 cols)


,carat,cut,color,clarity,depth,table,price,x,y,z,volume,density,xy_ratio,log_carat,cut_rank,color_rank,clarity_rank
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43,38.202030,0.006021,0.992462,0.207014,4,5,1
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31,34.505856,0.006086,1.013021,0.190620,3,5,2
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31,38.076885,0.006040,0.995086,0.207014,1,5,4
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63,46.724580,0.006207,0.992908,0.254642,3,1,3
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75,51.917250,0.005971,0.997701,0.270027,1,0,1


## Encoding — and why this dataset is the *opposite* of the last oneIn the insurance clustering project, `region` and `sex` had **no natural order**, soone-hot encoding was mandatory: any integer mapping would have invented a fake ranking.Here, `cut`, `color`, and `clarity` are **quality grades**. Ideal genuinely *is* better thanPremium, which genuinely *is* better than Very Good. That ordering is real, it is what a GIAgrader certifies, and it is information.So we use **ordinal encoding** — and this time it is not a shortcut, it's the correct choice.One-hot would *destroy* the ordering and force the network to rediscover it from scratch outof 20 disconnected binary columns.**The rule: one-hot when categories are unordered. Ordinal when the order is real.**Look at the data and decide; don't apply a reflex.

In [3]:
print("cut     ", CUT_MAP)
print("color   ", COLOR_MAP)
print("clarity ", CLARITY_MAP)

display(df[["cut", "cut_rank", "color", "color_rank", "clarity", "clarity_rank"]].head())

cut      {'Fair': 0, 'Good': 1, 'Very Good': 2, 'Premium': 3, 'Ideal': 4}
color    {'J': 0, 'I': 1, 'H': 2, 'G': 3, 'F': 4, 'E': 5, 'D': 6}
clarity  {'I1': 0, 'SI2': 1, 'SI1': 2, 'VS2': 3, 'VS1': 4, 'VVS2': 5, 'VVS1': 6, 'IF': 7}


,cut,cut_rank,color,color_rank,clarity,clarity_rank
0,Ideal,4,E,5,SI2,1
1,Premium,3,E,5,SI1,2
2,Good,1,E,5,VS1,4
3,Premium,3,I,1,VS2,3
4,Good,1,J,0,SI2,1


## Engineered features

In [4]:
display(df[["carat", "x", "y", "z", "volume", "density", "xy_ratio", "log_carat"]].head())
print(df[["volume", "density", "xy_ratio", "log_carat"]].describe().round(3).T)

,carat,x,y,z,volume,density,xy_ratio,log_carat
0,0.23,3.95,3.98,2.43,38.202030,0.006021,0.992462,0.207014
1,0.21,3.89,3.84,2.31,34.505856,0.006086,1.013021,0.190620
2,0.23,4.05,4.07,2.31,38.076885,0.006040,0.995086,0.207014
3,0.29,4.20,4.23,2.63,46.724580,0.006207,0.992908,0.254642
4,0.31,4.34,4.35,2.75,51.917250,0.005971,0.997701,0.270027


             count     mean     std     min     25%      50%      75%      max
volume     53766.0  129.772  76.359  31.708  65.206  114.851  170.831  790.133
density    53766.0    0.006   0.000   0.003   0.006    0.006    0.006    0.023
xy_ratio   53766.0    0.999   0.010   0.749   0.993    0.996    1.007    1.616
log_carat  53766.0    0.555   0.245   0.182   0.336    0.531    0.713    1.793


- **`volume` = x·y·z** — carat is a *weight*, volume is a *size*. A poorly cut stone is bulky  for its weight. Giving the model both lets it learn the gap between them.- **`density` = carat / volume** — high means the stone is dense for its size: a deep,  weight-retaining cut.- **`xy_ratio` = x / y** — a round brilliant should be circular, so x ≈ y. Deviation from 1.0  is a measurable symmetry defect that no raw column captures.- **`log_carat`** — straightens the multiplicative carat–price relationship we saw in the EDA.

## Feature selection — and the leakage trap**The three tasks get three different feature sets.** This is not fussiness; getting it wrongproduces a model that scores brilliantly and is worth nothing.

In [5]:
from src.data_prep import CLF_FEATURES, REG_FEATURES, CLU_FEATURES

print("CLASSIFICATION (predict clarity):")
print("  ", CLF_FEATURES)
print("   [price INCLUDED - a listed stone has a known price]")
print("   [clarity_rank EXCLUDED - that IS the answer]\n")

print("REGRESSION (predict price):")
print("  ", REG_FEATURES)
print("   [clarity_rank INCLUDED - a graded stone's clarity is known before pricing]")
print("   [price / log_price / price_per_carat EXCLUDED - see below]\n")

print("CLUSTERING (segment inventory):")
print("  ", CLU_FEATURES)
print("   [no target at all - unsupervised]")

CLASSIFICATION (predict clarity):
   ['carat', 'log_carat', 'depth', 'table', 'price', 'x', 'y', 'z', 'volume', 'density', 'xy_ratio', 'cut_rank', 'color_rank']
   [price INCLUDED - a listed stone has a known price]
   [clarity_rank EXCLUDED - that IS the answer]

REGRESSION (predict price):
   ['carat', 'log_carat', 'depth', 'table', 'x', 'y', 'z', 'volume', 'density', 'xy_ratio', 'cut_rank', 'color_rank', 'clarity_rank']
   [clarity_rank INCLUDED - a graded stone's clarity is known before pricing]
   [price / log_price / price_per_carat EXCLUDED - see below]

CLUSTERING (segment inventory):
   ['carat', 'price', 'cut_rank', 'color_rank', 'clarity_rank', 'volume']
   [no target at all - unsupervised]


### The `price_per_carat` trapIt is tempting to engineer `price_per_carat = price / carat`. It's a real gemmologicalquantity and it looks like a useful feature.**Feed it to the price regressor and you get R² = 1.00.** Because `price_per_carat × carat`*is* `price`, exactly. The model hasn't learned anything about diamonds; it has learned tomultiply.And at prediction time you don't *have* price_per_carat — you'd need the price to compute it,which is the thing you're trying to predict. The model is unusable the moment it leaves yournotebook.**This is data leakage, and it is the most common way a machine learning project failssilently.** The tell is always the same: *a score that seems too good.* When your R² comes backat 0.999, do not celebrate. Go looking for the leak.The test: **would I actually have this value, at the moment I need to make the prediction?**If no, it is not a feature.

## Splitting and scaling

In [6]:
from src.data_prep import split_scaled

X_tr, X_val, X_te, y_tr, y_val, y_te, scaler = split_scaled(
    df, REG_FEATURES, "price", test_size=0.2, val_size=0.1
)

print(f"train {X_tr.shape}   val {X_val.shape}   test {X_te.shape}")
print(f"\ntrain mean ~0: {X_tr.mean(axis=0)[:4].round(4)}")
print(f"test  mean:    {X_te.mean(axis=0)[:4].round(4)}   <- NOT exactly 0, and that is correct")

train (37636, 13)   val (5376, 13)   test (10754, 13)

train mean ~0: [-0. -0. -0. -0.]
test  mean:    [ 0.0005  0.0047 -0.0036  0.0063]   <- NOT exactly 0, and that is correct


**Two things to notice.****A three-way split, not two.** Train / validation / test. The *validation* set is whatEarlyStopping watches while the model trains. If early stopping watched the *test* set, youwould be tuning against it — and the test set would no longer be held out. It would just be asecond validation set wearing a disguise, and your final number would be optimistic.**The scaler is fit on train only.** `fit_transform(X_train)` then `transform(X_val)` and`transform(X_test)`. Notice the test set's scaled mean is *not* exactly zero — and that isexactly right. It is being measured against the *training* distribution, which is the onlydistribution the model has ever been allowed to see. If you fit the scaler on everything, thetest data's own statistics leak into training and every score you report is inflated.**The scaler is part of the model.** It must be saved and shipped alongside it.